# Capa 1 - Pipeline RAG (Retrieval-Augmented Generation)

## 1.Imports y configuración


In [1]:
# pip install --upgrade pymongo

In [2]:
#pip install faiss-cpu

In [3]:
import pandas as pd
import sys, os
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import faiss

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from data.corpus_canciones import get_collection
from src.rag_utils import chunking_por_estrofa, chunking_cancion_completa, mostrar_metricas_chunking, generar_o_cargar_embeddings,crear_indice_faiss, buscar_chunks_relevantes,cargar_modelo, generar_con_flan_t5, rag_completo, sin_rag
print("Funciona correctamente")

C:\PF-Chatbot-Musical\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Funciona correctamente


In [4]:
# Obtener la colección
collection = get_collection()

# Cargar
cursor = collection.find({}, {"_id": 0, "artista": 1, "nombre": 1, "letra": 1, "genero": 1, "anio":1, "titulo":1})
df_canciones = pd.DataFrame(list(cursor))

print(f"Total de canciones cargadas: {len(df_canciones)}")
display(df_canciones.head())

Total de canciones cargadas: 10384


,titulo,artista,genero,anio,letra
0,The Chain - 2004 Remaster,Fleetwood Mac,rock,1977,Fuck Listen to the wind blow Watch the sun ris...
1,Everlong,Foo Fighters,rock,1997,Hello I've waited here for you Everlong Tonigh...
2,"Lover, You Should've Come Over",Jeff Buckley,rock,1994,Looking out the door I see the rain Fall upon ...
3,Iris,Goo Goo Dolls,rock,2023,And I'd give up forever to touch you 'Cause I ...
4,Still Into You,Paramore,rock,2013,Can't count the years on one hand that we've b...


## 2. Chunking

Se aplicaron dos estrategias para la fragmentación del texto

### 2.1 Chunking por estrofa

In [5]:
# Convierte el DataFrame a una lista de diccionarios
datos_para_procesar = df_canciones.to_dict('records')

# Ejecucion de la función creada chunking_musical()
fragmentos_procesados_A = chunking_por_estrofa(datos_para_procesar)

### 2.2 Chunking por cancion completa

In [6]:
fragmentos_procesados_B = chunking_cancion_completa(datos_para_procesar)

###  2.3 Metricas Chunking por estrofa y Chunking por cancion completa

In [7]:
# Comparación de metricas
mostrar_metricas_chunking(fragmentos_procesados_A, "A (Por Estrofas)")
mostrar_metricas_chunking(fragmentos_procesados_B, "B (Canción Completa)")

--- MÉTRICAS ESTRATEGIA: A (Por Estrofas) ---
Total chunks: 24285
Tamaño promedio: 753 caracteres
Min/Max: 65/1492 caracteres
Ejemplo: 'Song: The Chain - 2004 Remaster | Artist: Fleetwood Mac | Genre: rock | Year: 1977
Lyrics: Fuck List...'
----------------------------------------

--- MÉTRICAS ESTRATEGIA: B (Canción Completa) ---
Total chunks: 10384
Tamaño promedio: 1647 caracteres
Min/Max: 150/51426 caracteres
Ejemplo: 'Song: The Chain - 2004 Remaster | Artist: Fleetwood Mac | Genre: rock | Year: 1977
Lyrics: Fuck List...'
----------------------------------------



## 3. Embeddings

#### Embeddings con la estrategia A: POR ESTROFA

In [8]:
# Cargamos el modelo multilingüe
print("Cargando modelo multilingüe...")
modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Desempaqueta en DOS variables:
vectores_A, chunks_A = generar_o_cargar_embeddings(fragmentos_procesados_A, "emb_por_estrofa", modelo_emb)


print(f"Embeddings cargados: shape = {vectores_A.shape}")
print(f"Chunks con texto cargados: {len(chunks_A)}")

Cargando modelo multilingüe...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4286.58it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cargando desde caché: emb_por_estrofa.pkl (Ahorrando tiempo...)
Embeddings cargados: shape = (24285, 384)
Chunks con texto cargados: 24285


#### Embeddings con la estrategia B: Canción Completa


In [9]:
# Cargamos el modelo multilingüe
print("Cargando modelo multilingüe...")
modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("\n--- ESTRATEGIA B: Cancion Completa ---")


# Desempaqueta en DOS variables:
vectores_B, chunks_B = generar_o_cargar_embeddings(fragmentos_procesados_B, "emb_cancion_completa", modelo_emb)

print(f"Embeddings cargados: shape = {vectores_B.shape}")

Cargando modelo multilingüe...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4016.76it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- ESTRATEGIA B: Cancion Completa ---
Cargando desde caché: emb_cancion_completa.pkl (Ahorrando tiempo...)
Embeddings cargados: shape = (10384, 384)


## 4. Creación indice FAISS

In [10]:
print("FAISS para Estrategia A...")
indice_A = crear_indice_faiss(vectores_A)

FAISS para Estrategia A...
Índice FAISS creado: 24285 vectores, dimensión 384


In [11]:
print("FAISS para Estrategia B...")
indice_B = crear_indice_faiss(vectores_B)

FAISS para Estrategia B...
Índice FAISS creado: 10384 vectores, dimensión 384


In [12]:
# Guardar índice FAISS para reutilizar en el chatbot
faiss.write_index(indice_A, "../notebooks/faiss_index_A.bin")
faiss.write_index(indice_B, "../notebooks/faiss_index_B.bin")
print("Índices FAISS guardados en disco.")

Índices FAISS guardados en disco.


## Busqueda Semantica

#### Chunks con estrategia A (POR ESTROFA)

In [13]:
pregunta = "songs about hate"

# Llamamos a la función pasando nuestras variables
mis_resultados = buscar_chunks_relevantes(
    pregunta=pregunta,
    indice_FAISS=indice_A,      # El índice FAISS
    chunks=fragmentos_procesados_A,  # chunking_por_estrofa
    modelo=modelo_emb,          # El modelo SentenceTransformer
    top_k=3                     # Los 3 mejores
)

Resultados para: 'songs about hate'

Resultado #1 (Similitud: 0.3430)
Song: My American Prayer | Artist: Downset | Genre: hip hop | Year: 1994
Lyrics: oubtless next in the death rate or love Suffocate beneath this fashion of hate I was taught to hate You and you weer taught to hate me love se...
--------------------------------------------------
Resultado #2 (Similitud: 0.3121)
Song: The Limit | Artist: Snowgoons,Viro The Virus | Genre: hip hop | Year: 2015
Lyrics: Human nature says the hate is in their blood Keep your friends close your enemies closer Y'all might find they're one and the same before it's...
--------------------------------------------------
Resultado #3 (Similitud: 0.3050)
Song: Golden Fleece | Artist: Hermit and the Recluse,Ka | Genre: hip hop | Year: 2018
Lyrics: ou that hate us is the half of you that need us I want compassion from the highest Food for the lowest Cures for the afflicted Rooves for the ...
--------------------------------------------------


#### Chunks con estrategia B (Cancion Completa)

In [14]:
pregunta = "songs about hate"

# Llamamos a la función pasando nuestras variables
mis_resultados = buscar_chunks_relevantes(
    pregunta=pregunta,
    indice_FAISS=indice_B,      # El índice FAISS
    chunks=fragmentos_procesados_A,  # chunking_por_estrofa
    modelo=modelo_emb,          # El modelo SentenceTransformer
    top_k=3                     # Queremos los 3 mejores
)

Resultados para: 'songs about hate'

Resultado #1 (Similitud: 0.2985)
Song: Last Night on Earth | Artist: Green Day | Genre: rock | Year: 2009
Lyrics: I text a postcard sent to you did it go through Sendin' all my love to you You are the moonlight of my life every night Givin' all my love to ...
--------------------------------------------------
Resultado #2 (Similitud: 0.2773)
Song: Driven Under | Artist: Seether | Genre: rock | Year: 2002
Lyrics: like she'd used it once before on him Then she told me she had a gun It sounded like she'd used it once before oh man We have to succumb to Th...
--------------------------------------------------
Resultado #3 (Similitud: 0.2769)
Song: The Stretch Armstrong and Bobbito Show on WKCR October 28 1993 | Artist: Nas,6'9,Jungle,Grand Wizard | Genre: hip hop | Year: 2014
Lyrics: lunt head Police Police want a nigga dead But I'm not goin' out like that black I kick the actual facts in solar Cold as a Polar Bear I swear ...
--------------------------

## 6. Generación de Respuestas

In [15]:
cargar_modelo()

Cargando google/flan-t5-base...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3603.60it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Modelo Base listo.


In [16]:
# Prueba
contexto_prueba = "The song 'Black Hole Sun' by Soundgarden was released in 1994 and is a grunge classic."
pregunta_prueba = "When was Black Hole Sun released?"

# Llamamos a la función
respuesta = generar_con_flan_t5(contexto_prueba, pregunta_prueba)

print(f"Respuesta del modelo: {respuesta}")

Respuesta del modelo: 1994


## 7. Sistema RAG completo

In [17]:
print("EJECUTANDO RAG - ESTRATEGIA A (ESTROFAS)")
rag_completo(
    pregunta="Give me a song about loneliness",
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

EJECUTANDO RAG - ESTRATEGIA A (ESTROFAS)

BUSCANDO: Give me a song about loneliness
Resultados para: 'Give me a song about loneliness'

Resultado #1 (Similitud: 0.4721)
Song: Loneliness Knows Me By Name | Artist: Westlife | Genre: pop | Year: 2024
Lyrics: he vacant space The cried out tears and a never ending maze Oh I have found what only loneliness provides A strength within knowing I will fin...
--------------------------------------------------
Resultado #2 (Similitud: 0.4310)
Song: Map of the Problematique | Artist: Muse | Genre: rock | Year: 2006
Lyrics: Fear and panic in the air I want to be free from desolation and despair And I feel like everything I sow Is being swept away well I refuse to ...
--------------------------------------------------
Resultado #3 (Similitud: 0.4294)
Song: Loneliness Knows Me By Name | Artist: Westlife | Genre: pop | Year: 2024
Lyrics: Oh yeah Loneliness is always looking for a friend It found me once And it has been around since then Loneliness is n

"Loneliness knows me by name Loneliness knows everything I keep inside My endless thoughts in the silence of the night Loneliness is the one who made me see Ain't nobody else Who can make a change but me no"

In [18]:
print("EJECUTANDO RAG - ESTRATEGIA B (Cancion completa)")
rag_completo(
    pregunta="Give me a song about loneliness",
    indice_faiss=indice_B,
    chunks=fragmentos_procesados_B,
    modelo_emb=modelo_emb,
    top_k=3
)

EJECUTANDO RAG - ESTRATEGIA B (Cancion completa)

BUSCANDO: Give me a song about loneliness
Resultados para: 'Give me a song about loneliness'

Resultado #1 (Similitud: 0.4310)
Song: Map of the Problematique | Artist: Muse | Genre: rock | Year: 2006
Lyrics: Fear and panic in the air I want to be free from desolation and despair And I feel like everything I sow Is being swept away well I refuse to ...
--------------------------------------------------
Resultado #2 (Similitud: 0.4294)
Song: Loneliness Knows Me By Name | Artist: Westlife | Genre: pop | Year: 2024
Lyrics: Oh yeah Loneliness is always looking for a friend It found me once And it has been around since then Loneliness is never waiting by the door I...
--------------------------------------------------
Resultado #3 (Similitud: 0.4277)
Song: Lost Friends | Artist: Middle Kids | Genre: pop | Year: 2018
Lyrics: Lonely is the sound when the truth hits the ground I lost all my friends that day I lost all my friends We were sitting 

'Genre: rock | Year: 1990s'

## Comparacion: RAG vs sin RAG

Pregunta 1: ¿Cuál es el sentimiento más común en las canciones?

In [19]:
pregunta_prueba = "¿What is the most common feeling of the songs?"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
happy

--- PRUEBA CON RAG ---

BUSCANDO: ¿What is the most common feeling of the songs?
Resultados para: '¿What is the most common feeling of the songs?'

Resultado #1 (Similitud: 0.3054)
Song: Make You Feel That Way | Artist: Blackalicious | Genre: hip hop | Year: 2002
Lyrics: u feel that way make you feel that way Make you feel that way You know it's like like the most greatest feeling you could ever feel y'know Lik...
--------------------------------------------------
Resultado #2 (Similitud: 0.2747)
Song: The Feeling | Artist: Justin Bieber,Halsey | Genre: hip hop | Year: 2015
Lyrics:  you Or am I in love with the feeling...
--------------------------------------------------
Resultado #3 (Similitud: 0.2313)
Song: Nearly Witches (Ever Since We Met...) | Artist: Panic! at the Disco | Genre: rock | Year: 2011
Lyrics:  only shoot up with your perfume It's the only thing that makes me feel as good as you do Ever since we met I've got just one regret to live t...
-

'elation'

Sin RAG el modelo responde "happy" — una respuesta que pudo ser valida pero en el contexto del corpus (genero, letra etc) es posible que predomine otro sentimiento.
Con RAG recupera canciones reales del corpus y ancla la respuesta en ellas,demostrando el valor de la recuperación semántica. Elation = Júbilo/Euforia

Pregunta 2: Dame una canción de hip hop que mencione el dinero o el éxito.

In [20]:
pregunta_prueba = "Give me a hip hop song that mentions money or success"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
i love you

--- PRUEBA CON RAG ---

BUSCANDO: Give me a hip hop song that mentions money or success
Resultados para: 'Give me a hip hop song that mentions money or success'

Resultado #1 (Similitud: 0.3781)
Song: Money | Artist: Internet Money,Don Toliver,Roddy Ricch | Genre: hip hop | Year: 2020
Lyrics: hat pussy in a minute So please don t call my phone if it ain t about them digits Why 'Cause I just want the money money money money money mon...
--------------------------------------------------
Resultado #2 (Similitud: 0.3775)
Song: Money Up | Artist: Rich Gang,Jacquees,Birdman,Caskey | Genre: hip hop | Year: 2016
Lyrics: I seen it all I done it all I did it I just need one time for the one time is you with it I'm still all out pipes when I come 'round in the ci...
--------------------------------------------------
Resultado #3 (Similitud: 0.3241)
Song: Throw It Away | Artist: 360 | Genre: hip hop | Year: 2012
Lyrics: Yeah ayo Money makes the world go round we

'The song "Taste It Away" was released in the United States on June 7, 2012.'

Sin el RAG no dio una respuesta coherente en cambio con el rag si se tuvo una respuesta mas satisfactoria y tomando en cuenta la letra el titulo de la canción incluso en un caso el nombre del artista/grupo

Pregunta 3: ¿Hay canciones de rock anteriores al año 1980 en el corpus?

In [21]:
pregunta_prueba = "¿Are there rock songs from before year 1980 in the corpus?"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
no

--- PRUEBA CON RAG ---

BUSCANDO: ¿Are there rock songs from before year 1980 in the corpus?
Resultados para: '¿Are there rock songs from before year 1980 in the corpus?'

Resultado #1 (Similitud: 0.2645)
Song: 1979 - Remastered 2012 | Artist: The Smashing Pumpkins | Genre: rock | Year: 2016
Lyrics: f a Bachelor Paramore Riot Phish A Picture of Nectar Pink Floyd The Dark Side the Moon The Wall Pixies Come On Pilgrim Surfer Rosa Doolittle T...
--------------------------------------------------
Resultado #2 (Similitud: 0.2413)
Song: All Along the Watchtower | Artist: Jimi Hendrix | Genre: rock | Year: 2021
Lyrics: orrow The Shirelles 152 Proud Mary Creedence Clearwater Revival 153 Super Freak Rick James 154 Spoonful Howlin Wolf 155 Last Nite The Strokes ...
--------------------------------------------------
Resultado #3 (Similitud: 0.2382)
Song: 1979 - Remastered 2012 | Artist: The Smashing Pumpkins | Genre: rock | Year: 2016
Lyrics: oston Bush Sixteen Stone Bu

'Jimi Hendrix'

Sin Rag directamente respondio "no" pero con el RAG fue una respuesta distinta si encontro información pero tomo en cuenta el nombre de la cancion y no el año ya que estos chunks se unio el nombre de la cancion, artista, genero y año pero apesar de eso no se concideraria del todo mal porque realizo una busqueda semantica decente.

Pregunta 4: Dame una canción sobre la soledad

In [22]:
pregunta_prueba = "Give me a song about loneliness"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
i can't believe i'm alone

--- PRUEBA CON RAG ---

BUSCANDO: Give me a song about loneliness
Resultados para: 'Give me a song about loneliness'

Resultado #1 (Similitud: 0.4721)
Song: Loneliness Knows Me By Name | Artist: Westlife | Genre: pop | Year: 2024
Lyrics: he vacant space The cried out tears and a never ending maze Oh I have found what only loneliness provides A strength within knowing I will fin...
--------------------------------------------------
Resultado #2 (Similitud: 0.4310)
Song: Map of the Problematique | Artist: Muse | Genre: rock | Year: 2006
Lyrics: Fear and panic in the air I want to be free from desolation and despair And I feel like everything I sow Is being swept away well I refuse to ...
--------------------------------------------------
Resultado #3 (Similitud: 0.4294)
Song: Loneliness Knows Me By Name | Artist: Westlife | Genre: pop | Year: 2024
Lyrics: Oh yeah Loneliness is always looking for a friend It found me once And it has been a

"Loneliness knows me by name Loneliness knows everything I keep inside My endless thoughts in the silence of the night Loneliness is the one who made me see Ain't nobody else Who can make a change but me no"

Sin Rag nos dio de resultado "i can't believe i'm alone" entendio la pregunta pero no dio una respuesta satisfactoria a diferencia de con RAG

Pregunta 5: Dame una canción de Linkin Park

In [23]:
pregunta_prueba = "Give me a song Linkin Park"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
i love you

--- PRUEBA CON RAG ---

BUSCANDO: Give me a song Linkin Park
Resultados para: 'Give me a song Linkin Park'

Resultado #1 (Similitud: 0.2849)
Song: Faint | Artist: Linkin Park | Genre: rock | Year: 2003
Lyrics: al Don't turn your back on me I won't be ignored...
--------------------------------------------------
Resultado #2 (Similitud: 0.2696)
Song: Kiss Me Kill Me | Artist: Frank Walker,Theresa Rex | Genre: hip hop | Year: 2021
Lyrics: Kiss me Down by the broken treehouse Swing me Upon its hanging tire Bring bring Bring your flowered hat We'll take the trail Marked on your fa...
--------------------------------------------------
Resultado #3 (Similitud: 0.2510)
Song: Lying from You | Artist: Linkin Park | Genre: rock | Year: 2003
Lyrics: you The very worst part of you is me...
--------------------------------------------------

Generando respuesta con modelo local...

RESPUESTA FINAL:
Lying from You


'Lying from You'

Pregunta 6: ¿Cuánto cuesta un vuelo a Madrid?

In [24]:
pregunta_prueba = "How much does a flight to Madrid cost?"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
£20

--- PRUEBA CON RAG ---

BUSCANDO: How much does a flight to Madrid cost?
Resultados para: 'How much does a flight to Madrid cost?'

Resultado #1 (Similitud: -0.5664)
Song: Paris | Artist: Lana Del Rey,SYML | Genre: pop | Year: 2023
Lyrics:  will wondering why Our time in paris Take me to movies and salons Paris make out in a dark restaurant Will dance until we die Wanna go out ev...
--------------------------------------------------
Resultado #2 (Similitud: -0.5787)
Song: MIA | Artist: Drake Bell | Genre: hip hop | Year: 2019
Lyrics: nkin' maybe we'll go MIA MIA Top floor of the Mandalay Fontainebleau to Montego Bay Thinkin' maybe we'll go MIA MIA Thinkin' maybe we'll go MI...
--------------------------------------------------
Resultado #3 (Similitud: -0.5798)
Song: On an Evening in Roma Sotter Cieolo de Roma | Artist: Dean Martin | Genre: pop | Year: 2024
Lyrics: Como e' bella ce' la luna brilla e' strette Strette como e' tutta bella a passeggiare Sotto il 

"MIA MIA Top floor of the Mandalay Fontainebleau to Montego Bay Thinkin' maybe we'll go MIA MIA Thinkin' maybe we'll go MIA MIA Margaritas in Cabo Black Murciélago Walk up to the plane land in Tokyo Sex in Spain where we take it slow Pop champagne wherever we go Yeah we on the loose Couldn't find us if they wanted to Bonnie and Clyde for"

Sin el RAG dio una respuesta que en otro contexto podria pasar por valida pero en este caso no y con RAG busco la manera de responder y busco relaciones como páises/capitales/estados/ciudades ejemplo Paris

Preguntas 7: ¿Cuál es la capital de Francia?

In [25]:
pregunta_prueba = "¿What is the capital of France?"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
london

--- PRUEBA CON RAG ---

BUSCANDO: ¿What is the capital of France?
Resultados para: '¿What is the capital of France?'

Resultado #1 (Similitud: -0.4066)
Song: Paris | Artist: Lana Del Rey,SYML | Genre: pop | Year: 2023
Lyrics:  will wondering why Our time in paris Take me to movies and salons Paris make out in a dark restaurant Will dance until we die Wanna go out ev...
--------------------------------------------------
Resultado #2 (Similitud: -0.4327)
Song: Darling Je Vous Aime Beaucoup | Artist: Dean Martin | Genre: pop | Year: 2013
Lyrics: Darling je vous aime beaucoup Je ne sais pas what you do You know you've completely Stolen my heart Morning noon and nighttime too Toujours wo...
--------------------------------------------------
Resultado #3 (Similitud: -0.4922)
Song: Knights Of The Cross | Artist: Savage | Genre: hip hop | Year: 2005
Lyrics: It is the end of the eleventh century The Christians rule European countries The Pope is the head of church

'Savage\'s song "Knights of the Cross" was released in 2005.'

Pregunta 8: ¿Cuál es la diferencia entre las letras de hip hop y rock?

In [26]:
pregunta_prueba = "¿What is the difference between hip hop and rock lyrics in the corpus?"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
Hip hop and rock lyrics

--- PRUEBA CON RAG ---

BUSCANDO: ¿What is the difference between hip hop and rock lyrics in the corpus?
Resultados para: '¿What is the difference between hip hop and rock lyrics in the corpus?'

Resultado #1 (Similitud: 0.2301)
Song: Difference Is | Artist: Lil Durk,Summer Walker | Genre: hip hop | Year: 2022
Lyrics: els when they deserve you and cherish these moments That's what he wanted 'Cause the difference is you ain't no fuck nigga and I'm feelin' it ...
--------------------------------------------------
Resultado #2 (Similitud: 0.1575)
Song: Pop Pop | Artist: Bruno Mars,Lupe Fiasco | Genre: hip hop | Year: 2022
Lyrics: welry all along...
--------------------------------------------------
Resultado #3 (Similitud: 0.1459)
Song: Difference Is | Artist: Lil Durk,Summer Walker | Genre: hip hop | Year: 2022
Lyrics: her another liter She steady runnin' pussy yummy cut on that Justin Bieber When I was nothin' gave her a hunnid I did a cou

"I'm a ride for my baby"

Pregunta 9: ¿Qué canción del corpus habla de "Mira cómo sale el sol, corre en las sombras"?

In [27]:
pregunta_prueba = "¿Which song in the corpus talks about 'Watch the sun rise Run in the shadows'?"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
saturday morning

--- PRUEBA CON RAG ---

BUSCANDO: ¿Which song in the corpus talks about 'Watch the sun rise Run in the shadows'?
Resultados para: '¿Which song in the corpus talks about 'Watch the sun rise Run in the shadows'?'

Resultado #1 (Similitud: 0.3679)
Song: Total Eclipse of the Sun | Artist: Nicki French | Genre: pop | Year: 1994
Lyrics: The beauty tender glow extinguished The sky dull from a breeze Ghostly the dawn without its red Uncanny stranged to our nature The light like ...
--------------------------------------------------
Resultado #2 (Similitud: 0.3368)
Song: Serenade | Artist: Emilie Kirsstadt | Genre: pop | Year: 2024
Lyrics: New world forming Picturesque in its stance Midnight calling Moonlight shadows start to dance For the dark finds ways of being Engraved in the...
--------------------------------------------------
Resultado #3 (Similitud: 0.3051)
Song: Sunspot | Artist: Dion | Genre: pop | Year: 2001
Lyrics: There s a sunspot in my eye

'Let the fire hatch out let the pain'

Pregunta 10(Prueba en español): Dame una canción de Red Hot Chili Peppers

In [29]:
pregunta_prueba = "Dame una canción de Red Hot Chili Peppers"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
el juego libre juegos en lnea

--- PRUEBA CON RAG ---

BUSCANDO: Dame una canción de Red Hot Chili Peppers
Resultados para: 'Dame una canción de Red Hot Chili Peppers'

Resultado #1 (Similitud: 0.3838)
Song: By the Way | Artist: Red Hot Chili Peppers | Genre: rock | Year: 2002
Lyrics: Standing in line to see the show tonight And there's a light on heavy glow By the way I tried to say I'd be there waiting for Dani the girl is...
--------------------------------------------------
Resultado #2 (Similitud: 0.3746)
Song: By the Way | Artist: Red Hot Chili Peppers | Genre: rock | Year: 2002
Lyrics: the way I tried to say I'd be there waiting for...
--------------------------------------------------
Resultado #3 (Similitud: 0.3637)
Song: Give It Away | Artist: Red Hot Chili Peppers | Genre: rock | Year: 1991
Lyrics: My mom I love her 'cause she love me Long gone are the times when she scrub me Feelin' good my brother gonna hug me Drinkin' my juice young lo...
----------

"Give it away give it away give it away now Give it away give it away give it away now Give it away give it away give it away give it away now Give it away give it away give it away give it away now I can't tell if I'm a kingpin or a pauper What I got you got to give it to your mamma What I got you've got to give it to your pappa What I got you"